In [1]:
!pip install -q datasets sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.5 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import textwrap
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
dataset = load_dataset("lavita/MedQuAD")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-e36383d177026d(…):   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})


In [5]:
df = dataset["train"].to_pandas()

In [6]:
df.columns

Index(['document_id', 'document_source', 'document_url', 'category',
       'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms',
       'question_id', 'question_focus', 'question_type', 'question', 'answer'],
      dtype='object')

In [7]:
df[["question", "answer", "document_source"]].isna().sum()

,0
question,0
answer,31034
document_source,0


In [8]:
df = df[["question", "answer", "document_source"]].dropna()
df.head()

,question,answer,document_source
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...,GHR
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...,GHR
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen...",GHR
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...,GHR
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...,GHR


In [9]:
len(df)

16407

In [10]:
documents = []

for idx, row in df.iterrows():
    text = f"""
Question: {row['question']}

Answer: {row['answer']}
"""

    documents.append({
        "id": idx,
        "text": text,
        "source": row["document_source"],
        "question": row["question"]
    })

len(documents)

16407

In [11]:
print(documents[0]["text"][:1000])


Question: What is (are) keratoderma with woolly hair ?

Answer: Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can include an abnorm

In [12]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
small_docs = documents

texts = [doc["text"] for doc in small_docs]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings.shape

Batches:   0%|          | 0/513 [00:00<?, ?it/s]

(16407, 384)

In [14]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 16407


In [15]:
def retrieve_docs(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        doc = small_docs[idx]
        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "distance": float(distance)
        })

    return results

In [16]:
query = "What are the symptoms of diabetes?"

results = retrieve_docs(query, top_k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc["source"])
    print("Matched Question:", doc["question"])
    print("Distance:", doc["distance"])
    print(doc["text"][:700])


--- Result 1 ---
Source: NIHSeniorHealth
Matched Question: What are the symptoms of Diabetes ?
Distance: 0.42415088415145874

Question: What are the symptoms of Diabetes ?

Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some lose weight without trying. Additional signs include sores that heal slowly, dry, itchy skin, loss of feeling or tingling in the feet and blurry eyesight. Some people with diabetes, however, have no symptoms at all.


--- Result 2 ---
Source: NIHSeniorHealth
Matched Question: What are the symptoms of Diabetes ?
Distance: 0.4445371627807617

Question: What are the symptoms of Diabetes ?

Answer: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million people in the United States have type 2 diabetes and dont know it, according to 

In [17]:
!pip install -q langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.4 MB/s eta 0:00:00


In [18]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

os.environ["GROQ_API_KEY"] = "Enter your API Key here"


In [19]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

In [20]:

def generate_answer(query, retrieved_docs):
    context = "\n\n".join([
        f"Source {i+1}:\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""
You are a medical document assistant.

Answer the user's question using ONLY the context provided below.

Rules:
1. Do not use outside knowledge.
2. If the context does not contain enough information, say:
   "The provided medical documents do not contain enough information."
3. Keep the answer simple and clear.
4. Do not diagnose the user.
5. Suggest consulting a healthcare professional when appropriate.

Context:
{context}

User question:
{query}

Answer:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [21]:
def rag_chatbot(query, top_k=3):
    retrieved_docs = retrieve_docs(query, top_k=top_k)
    answer = generate_answer(query, retrieved_docs)

    print("Question:")
    print(query)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n--- Source {i} ---")
        print("Dataset Source:", doc["source"])
        print("Matched Question:", doc["question"])
        print("Distance:", doc["distance"])

In [22]:
rag_chatbot("How is high blood pressure treated?")

Question:
How is high blood pressure treated?

Answer:
High blood pressure is treated with lifestyle changes and medicines. Treatment can help control blood pressure, but it will not cure high blood pressure, even if your blood pressure readings appear normal. 

Lifestyle changes include making ongoing medical care a priority. 

Medicines work in different ways, such as removing extra fluid and salt from your body, slowing down the heartbeat, or relaxing and widening blood vessels. Some common types of medicines used to treat high blood pressure include:

- Diuretics (water or fluid Pills)
- Beta Blockers
- Angiotensin-Converting Enzyme (ACE) Inhibitors
- Angiotensin II Receptor Blockers (ARBs)
- Calcium Channel Blockers
- Alpha Blockers
- Alpha-Beta Blockers
- Central Acting Agents
- Vasodilators

It's recommended to work with your healthcare team for lifelong blood pressure control.

Retrieved Sources:

--- Source 1 ---
Dataset Source: NIHSeniorHealth
Matched Question: What are the t